# 🎬 Hook Analysis — Google Drive VTT Parser

This notebook parses `.vtt` caption files from Google Drive (`Casa Siete/transcripts`), extracts the **0-4 second hook transcript** (last word inclusive) and the **full clean transcript**, deduplicates YouTube rolling auto-captions while preserving spoken emphasis word repetitions, and exports the dataset to a local CSV (`hooks_analysis.csv`).

### Output Format (CSV)
- `id`: YouTube Video ID (extracted from filename)
- `hook_transcript`: Clean transcript of words spoken in the first 0.0 to 4.0 seconds (last word inclusive)
- `full_transcript`: Full deduplicated video transcript


## 📁 Step 1: Mount Google Drive

Mount Google Drive to access files under `/content/drive/MyDrive/Casa Siete/transcripts`.

In [6]:
#@title Mount Google Drive { display-mode: "form" }
import os

IS_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    print("✅ Google Drive successfully mounted.")
except ImportError:
    print("ℹ️ Running outside Google Colab environment. Operating in local mode.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive successfully mounted.


## ⚙️ Step 2: Configuration & Parameters

In [7]:
#@title Configuration Parameters { display-mode: "form" }
transcripts_dir = "/content/drive/MyDrive/Casa Siete/Transcripts" #@param {type:"string"}
output_csv_path = "hooks_analysis.csv" #@param {type:"string"}
SELECTED_VIDEO_ID = "sample_video" #@param {type:"string"}

print(f"📌 Transcripts Directory: {transcripts_dir}")
print(f"📌 Output CSV Path: {output_csv_path}")
print(f"📌 Selected Verification Video ID: {SELECTED_VIDEO_ID}")

📌 Transcripts Directory: /content/drive/MyDrive/Casa Siete/Transcripts
📌 Output CSV Path: hooks_analysis.csv
📌 Selected Verification Video ID: sample_video


## 🧹 Step 3: VTT Parser Algorithm with Emphasis Word Preservation

In [8]:
import os
import re
import glob

def parse_vtt_timestamp(ts_str):
    clean = ts_str.strip().replace(',', '.')
    parts = clean.split(':')
    if len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    elif len(parts) == 2:
        return int(parts[0]) * 60 + float(parts[1])
    return float(parts[0])

def normalize_word(word):
    return re.sub(r'[^\w]', '', word).lower()

def extract_cue_timed_words(cue_start, raw_text):
    clean_tags = re.sub(r'</?[a-zA-Z][^>]*>', '', raw_text)
    pattern = re.compile(r'(<(?:\d{2}:)?\d{2}:\d{2}[\.,]\d{3}>)|([^\s<>]+)')

    current_time = cue_start
    timed_words = []

    for match in pattern.finditer(clean_tags):
        ts_tag, word = match.groups()
        if ts_tag:
            current_time = parse_vtt_timestamp(ts_tag.strip('<>'))
        elif word:
            timed_words.append({'word': word, 'start': current_time})

    return timed_words

def parse_and_clean_vtt(vtt_content):
    lines = vtt_content.splitlines()
    cues = []
    current_cue = None
    ts_pattern = re.compile(r'(\d{2}:)?\d{2}:\d{2}[\.,]\d{3}\s*-->\s*(\d{2}:)?\d{2}:\d{2}[\.,]\d{3}')

    for line in lines:
        line_str = line.strip()
        if not line_str or line_str.startswith('WEBVTT') or line_str.startswith('NOTE') or line_str.startswith('STYLE'):
            continue

        if ts_pattern.search(line_str):
            parts = line_str.split('-->')
            start = parse_vtt_timestamp(parts[0].strip())
            end = parse_vtt_timestamp(parts[1].strip().split()[0])
            current_cue = {'start': start, 'end': end, 'raw': []}
            cues.append(current_cue)
        elif current_cue is not None and not line_str.isdigit():
            current_cue['raw'].append(line_str)

    cumulative_timed_words = []
    last_cue_end = 0.0

    for cue in cues:
        raw_text = ' '.join(cue['raw'])
        cue_words = extract_cue_timed_words(cue['start'], raw_text)
        if not cue_words:
            continue

        norm_cue_words = [normalize_word(tw['word']) for tw in cue_words]
        max_overlap = 0

        # 1. Word-level suffix-prefix overlap search against recent cumulative transcript
        max_lookback = min(len(cumulative_timed_words), 50)
        for k in range(min(len(cue_words), max_lookback), 0, -1):
            prefix = norm_cue_words[:k]
            suffix = [normalize_word(tw['word']) for tw in cumulative_timed_words[-k:]]
            if prefix == suffix:
                max_overlap = k
                break

        # 2. Sub-phrase flash duplicate cue check ONLY if cue start is contiguous with last cue (<= last_cue_end + 0.2s)
        if max_overlap == 0 and len(cumulative_timed_words) > 0:
            if cue['start'] <= last_cue_end + 0.2:
                norm_cum = [normalize_word(tw['word']) for tw in cumulative_timed_words[-max_lookback:]]
                for i in range(len(norm_cum) - len(norm_cue_words) + 1):
                    if norm_cum[i:i+len(norm_cue_words)] == norm_cue_words:
                        max_overlap = len(norm_cue_words)
                        break

        new_words = cue_words[max_overlap:]
        cumulative_timed_words.extend(new_words)
        last_cue_end = max(last_cue_end, cue['end'])

    hook_words = [tw['word'] for tw in cumulative_timed_words if tw['start'] <= 4.0]
    full_words = [tw['word'] for tw in cumulative_timed_words]

    return {
        'hook_transcript': ' '.join(hook_words),
        'full_transcript': ' '.join(full_words),
        'timed_words': cumulative_timed_words,
        'cues': cues
    }

## 📂 Step 4: Batch Processing VTT Directory & Export to CSV

In [9]:
try:
    import pandas as pd
except ImportError:
    import sys
    !{sys.executable} -m pip install pandas
    import pandas as pd

def process_transcripts_directory(directory_path, output_csv):
    records = []
    search_path = directory_path
    if not os.path.exists(search_path):
        print(f"⚠️ Directory '{search_path}' not found. Checking local fallback directory...")
        search_path = "./transcripts"
        os.makedirs(search_path, exist_ok=True)
        sample_vtt_file = os.path.join(search_path, "sample_video.vtt")
        if not os.path.exists(sample_vtt_file) and os.path.exists("../clip-cli/sample.vtt"):
            with open("../clip-cli/sample.vtt", "r", encoding="utf-8") as sf:
                with open(sample_vtt_file, "w", encoding="utf-8") as df:
                    df.write(sf.read())

    vtt_files = sorted(glob.glob(os.path.join(search_path, "*.vtt")))
    print(f"📂 Found {len(vtt_files)} .vtt files in '{search_path}'.")

    processed_data = {}

    for fpath in vtt_files:
        video_id = os.path.splitext(os.path.basename(fpath))[0]
        with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
            vtt_content = f.read()

        parsed = parse_and_clean_vtt(vtt_content)
        records.append({
            'id': video_id,
            'hook_transcript': parsed['hook_transcript'],
            'full_transcript': parsed['full_transcript']
        })
        processed_data[video_id] = {
            'filepath': fpath,
            'parsed': parsed,
            'raw_content': vtt_content
        }

    df = pd.DataFrame(records, columns=['id', 'hook_transcript', 'full_transcript'])
    df.to_csv(output_csv, index=False, encoding='utf-8')
    print(f"✅ Exported {len(df)} records to local CSV: '{output_csv}'")

    return df, processed_data

df_results, processed_cache = process_transcripts_directory(transcripts_dir, output_csv_path)
if not df_results.empty:
    print(df_results.head(10))


📂 Found 12 .vtt files in '/content/drive/MyDrive/Casa Siete/Transcripts'.
✅ Exported 12 records to local CSV: 'hooks_analysis.csv'
            id                                    hook_transcript  \
0  3qVlai0kH2k  Es bien difícil para un hombre atender las emo...   
1  Dy979xqrjzk  No es pedir, pedir, pedir. Tú dame y tú haz y ...   
2  FOOmc_TPG9w  No hay mejor mujer que tú siente que sigue siendo   
3  JlXmJUd1Qd4  Cuando una pareja dice, "Nunca hemos tenido un...   
4  Kh-F1SmNn9o      Tu emoción es tuya, no es de la otra persona.   
5  csd_lZKALgA      Cuando queremos romance, lo ponemos de cierta   
6  jgLSbzYQ1O0      Los problemas que tenemos con nuestra pareja,   
7  k718ulBdloM  Quiero que me invites a cenar. Quiero que adiv...   
8  kTYK8ewdmhA  Cuando no es real la autoestima, la confianza ...   
9  lRZnXeQoI8U  Una mujer siempre quiere ser atendida. Somos u...   

                                     full_transcript  
0  Es bien difícil para un hombre atender las emo...  

## 🔍 Step 5: Verification Sample for One Video

In [10]:
from IPython.display import HTML, display

def verify_video_sample(video_id, cache):
    if video_id not in cache:
        available_ids = list(cache.keys())
        if available_ids:
            print(f"⚠️ Video ID '{video_id}' not found in cache. Defaulting to available ID: '{available_ids[0]}'")
            video_id = available_ids[0]
        else:
            print("⚠️ No processed video records available for verification sample.")
            return

    data = cache[video_id]
    parsed = data['parsed']

    html_out = f"""
    <h3>🔍 Verification Sample for Video ID: <code style="color:#0066cc;">{video_id}</code></h3>
    <p><b>File Path:</b> <code>{data['filepath']}</code></p>
    <div style="display: flex; gap: 20px; flex-wrap: wrap;">
      <div style="flex: 1; min-width: 300px; border: 1px solid #ccc; padding: 12px; border-radius: 6px; background-color: #f7f9fa;">
        <h4 style="margin-top:0; color:#d9534f;">🎯 Hook Transcript (0s - 4s, last word inclusive)</h4>
        <p style="font-size: 14px; font-weight: bold; color: #111;">{parsed['hook_transcript'] or '<i>[Empty]</i>'}</p>
      </div>
      <div style="flex: 2; min-width: 300px; border: 1px solid #ccc; padding: 12px; border-radius: 6px; background-color: #f7f9fa;">
        <h4 style="margin-top:0; color:#28a745;">📜 Full Clean Transcript</h4>
        <p style="font-size: 13px; color: #333; max-height: 200px; overflow-y: auto;">{parsed['full_transcript'] or '<i>[Empty]</i>'}</p>
      </div>
    </div>
    """
    display(HTML(html_out))

    print(f"\n⏱️ Timed Words Word-by-Word Inspection (First 6 seconds for video '{video_id}'):")
    first_6s_words = [tw for tw in parsed['timed_words'] if tw['start'] <= 6.0]
    df_words = pd.DataFrame(first_6s_words)
    if not df_words.empty:
        df_words['in_hook_0_4s'] = df_words['start'].apply(lambda t: "✅ Yes" if t <= 4.0 else "❌ No")
        display(df_words)

verify_video_sample(SELECTED_VIDEO_ID, processed_cache)


⚠️ Video ID 'sample_video' not found in cache. Defaulting to available ID: '3qVlai0kH2k'



⏱️ Timed Words Word-by-Word Inspection (First 6 seconds for video '3qVlai0kH2k'):


,word,start,in_hook_0_4s
0,Es,0.120,✅ Yes
1,bien,0.280,✅ Yes
2,difícil,0.560,✅ Yes
3,para,1.240,✅ Yes
4,un,1.520,✅ Yes
5,hombre,1.719,✅ Yes
6,atender,2.760,✅ Yes
7,las,3.159,✅ Yes
8,emociones,3.399,✅ Yes
9,de,3.840,✅ Yes
